In [121]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from ucimlrepo import fetch_ucirepo
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
from tensorflow import keras
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [79]:
bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features
y = bank_marketing.data.targets

In [80]:
X = X.select_dtypes(include=[np.number]).fillna(0)
y = pd.Series(y.iloc[:, 0]).apply(lambda x: 1 if x == 'yes' else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [118]:
class MyTree:
    def __init__(self, max_depth=5, class_weight=None):
        self.max_depth = max_depth
        self.class_weight = class_weight
        self.tree = None
        self.features = None
        self.class_weights = None

    def fit(self, X, y):
        X, y = np.array(X), np.array(y).flatten()
        self.n_features = X.shape[1]

        if self.class_weight == 'balanced':
            unique, counts = np.unique(y, return_counts=True)
            n_samples = len(y)
            self.class_weights = {cls: n_samples / (len(unique) * count) for cls, count in zip(unique, counts)}
        else:
            self.class_weights = {0: 1.0, 1: 1.0}

        if self.features is None:
            self.features = np.arange(self.n_features)
        self.tree = self._build(X, y, 0)

    def _build(self, X, y, depth):
        if depth >= self.max_depth or len(np.unique(y)) == 1 or len(y) < 2:
            return self._leaf_value(y)

        best_feat, best_thr = self._best_split(X, y)
        if best_feat is None:
            return self._leaf_value(y)

        left_mask = X[:, best_feat] <= best_thr
        right_mask = X[:, best_feat] > best_thr

        if sum(left_mask) == 0 or sum(right_mask) == 0:
            return self._leaf_value(y)

        return {
            'feat': best_feat,
            'thr': best_thr,
            'left': self._build(X[left_mask], y[left_mask], depth + 1),
            'right': self._build(X[right_mask], y[right_mask], depth + 1)
        }

    def _leaf_value(self, y):
        if len(y) == 0:
            return 0

        unique, counts = np.unique(y, return_counts=True)
        if self.class_weight == 'balanced':
            weighted_counts = [counts[i] * self.class_weights[unique[i]] for i in range(len(unique))]
            return unique[np.argmax(weighted_counts)]
        else:
            return unique[np.argmax(counts)]

    def _best_split(self, X, y):
        best_gain = -1
        best_feat, best_thr = None, None

        for f in self.features:
            unique_thresholds = np.unique(X[:, f])
            if len(unique_thresholds) > 100:
                unique_thresholds = np.percentile(X[:, f], np.linspace(0, 100, 100))

            for thr in unique_thresholds:
                left_mask = X[:, f] <= thr
                right_mask = X[:, f] > thr
                if sum(left_mask) == 0 or sum(right_mask) == 0:
                    continue
                gain = self._gain(y, y[left_mask], y[right_mask])
                if gain > best_gain:
                    best_gain = gain
                    best_feat = f
                    best_thr = thr
        return best_feat, best_thr

    def _gain(self, y, yl, yr):
        p = len(yl) / len(y)
        return self._impurity(y) - p * self._impurity(yl) - (1-p) * self._impurity(yr)

    def _impurity(self, y):
        if len(y) == 0:
            return 0

        unique, counts = np.unique(y, return_counts=True)
        if self.class_weight == 'balanced':
            weighted_counts = [counts[i] * self.class_weights[unique[i]] for i in range(len(unique))]
            probs = np.array(weighted_counts) / np.sum(weighted_counts)
        else:
            probs = counts / len(y)
        return 1 - np.sum(probs ** 2)

    def predict(self, X):
        X = np.array(X)
        return np.array([self._predict_row(x, self.tree) for x in X])

    def _predict_row(self, x, node):
        if not isinstance(node, dict):
            return node
        if x[node['feat']] <= node['thr']:
            return self._predict_row(x, node['left'])
        return self._predict_row(x, node['right'])

In [119]:
class MyRandomForest:
    def __init__(self, n_trees=10, max_depth=5, max_features='sqrt', class_weight='balanced'):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.max_features = max_features
        self.class_weight = class_weight
        self.trees = []

    def fit(self, X, y):
        X, y = np.array(X), np.array(y).flatten()
        n_samples = X.shape[0]
        n_features = X.shape[1]

        if self.max_features == 'sqrt':
            features_per_tree = int(np.sqrt(n_features))
        else:
            features_per_tree = n_features

        self.trees = []
        for _ in range(self.n_trees):
            idx = np.random.choice(n_samples, n_samples, replace=True)
            X_sample, y_sample = X[idx], y[idx]

            tree = MyTree(max_depth=self.max_depth, class_weight=self.class_weight)
            tree.features = np.random.choice(X.shape[1], features_per_tree, replace=False)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        X = np.array(X)
        predictions = np.array([tree.predict(X) for tree in self.trees])
        result = np.zeros(len(X), dtype=np.int32)
        for i in range(len(X)):
            votes = predictions[:, i]
            unique, counts = np.unique(votes, return_counts=True)
            if 1 in unique:
                idx_1 = np.where(unique == 1)[0][0]
                if counts[idx_1] >= np.ceil(len(votes) / 2):
                    result[i] = 1
                else:
                    result[i] = unique[np.argmax(counts)]
            else:
                result[i] = unique[np.argmax(counts)]
        return result

In [98]:
rf = MyRandomForest(n_trees=10, max_depth=5, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))


Accuracy: 0.7496
              precision    recall  f1-score   support

           0       0.95      0.76      0.84      7985
           1       0.27      0.67      0.39      1058

    accuracy                           0.75      9043
   macro avg       0.61      0.72      0.61      9043
weighted avg       0.87      0.75      0.79      9043



In [115]:
class MyRandomForestReg:
    def __init__(self, n_trees=10, max_depth=5, max_features='sqrt'):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.max_features = max_features
        self.trees = []

    def fit(self, X, y):
        X, y = np.array(X), np.array(y).flatten()
        n_samples = X.shape[0]
        self.n_features = X.shape[1]

        self.features_per_tree = int(np.sqrt(self.n_features)) if self.max_features == 'sqrt' else self.n_features

        for _ in range(self.n_trees):
            idx = np.random.choice(n_samples, n_samples, replace=True)
            X_sample = X[idx]
            y_sample = y[idx]

            tree = MyTree(max_depth=self.max_depth)
            tree.n_features = self.features_per_tree
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        X = np.array(X)
        predictions = np.array([tree.predict(X) for tree in self.trees])
        return np.mean(predictions, axis=0)

In [116]:
rf = MyRandomForestReg(n_trees=10, max_depth=5)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f"R2: {r2_score(y_test, y_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")

R2: 0.0563
MSE: 0.0975


In [106]:
class MyGradientBoosting:
    def __init__(self, n_estimators=50, learning_rate=0.1, max_depth=3, class_weight=None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.class_weight = class_weight
        self.trees = []
        self.base_pred = None

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def fit(self, X, y):
        X, y = np.array(X), np.array(y).flatten()
        p = np.mean(y)
        self.base_pred = np.log(p / (1 - p))
        F = np.full(len(y), self.base_pred)
        residuals = y - self._sigmoid(F)

        for _ in range(self.n_estimators):
            tree = MyTree(max_depth=self.max_depth, class_weight=self.class_weight)
            tree.fit(X, residuals)
            self.trees.append(tree)

            F += self.learning_rate * tree.predict(X)
            residuals = y - self._sigmoid(F)

    def predict_proba(self, X):
        X = np.array(X)
        F = np.full(len(X), self.base_pred)
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        return self._sigmoid(F)

    def predict(self, X):
        return (self.predict_proba(X) > 0.5).astype(int)

In [107]:
gb = MyGradientBoosting(n_estimators=20, learning_rate=0.1, max_depth=3, class_weight='balanced')
gb.fit(X_train, y_train)
y_pred = gb.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

Accuracy: 0.8830
              precision    recall  f1-score   support

           0       0.88      1.00      0.94      7985
           1       0.00      0.00      0.00      1058

    accuracy                           0.88      9043
   macro avg       0.44      0.50      0.47      9043
weighted avg       0.78      0.88      0.83      9043



In [108]:
class MyGradientBoostingReg:
    def __init__(self, n_estimators=50, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.base_pred = None

    def fit(self, X, y):
        X, y = np.array(X), np.array(y).flatten()
        self.base_pred = np.mean(y)

        F = np.full(len(y), self.base_pred)
        residuals = y - F

        for _ in range(self.n_estimators):
            tree = MyTree(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)

            F += self.learning_rate * tree.predict(X)
            residuals = y - F

    def predict(self, X):
        X = np.array(X)
        F = np.full(len(X), self.base_pred)
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        return F

In [110]:
gb = MyGradientBoostingReg(n_estimators=20, learning_rate=0.1, max_depth=3)
gb.fit(X_train, y_train)
y_pred = gb.predict(X_test)

print(f"R2: {r2_score(y_test, y_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")

R2: -0.0247
MSE: 0.1059


In [120]:
model = keras.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

y_pred = (model.predict(X_test) > 0.5).astype(int)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

Epoch 1/50
905/905 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8863 - loss: 0.2912 - val_accuracy: 0.8898 - val_loss: 0.2574
Epoch 2/50
905/905 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8899 - loss: 0.2703 - val_accuracy: 0.8902 - val_loss: 0.2553
Epoch 3/50
905/905 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8886 - loss: 0.2664 - val_accuracy: 0.8911 - val_loss: 0.2541
Epoch 4/50
905/905 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8900 - loss: 0.2640 - val_accuracy: 0.8889 - val_loss: 0.2620
Epoch 5/50
905/905 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8897 - loss: 0.2639 - val_accuracy: 0.8911 - val_loss: 0.2520
Epoch 6/50
905/905 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8908 - loss: 0.2618 - val_accuracy: 0.8929 - val_loss: 0.2516
Epoch 7/50
905/905 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8909 - loss: 0.2609 - val_accuracy: 0.8915 - val_loss: 0.2520
Epoch 8/50
905/905 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8922 - loss: 0.2598 - val_accuracy: 0.

In [126]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train.tolist(), dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test.tolist(), dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

class PyTorchNNReg(nn.Module):
    def __init__(self, input_size):
        super(PyTorchNNReg, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 1)
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

model = PyTorchNNReg(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
for epoch in range(epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    y_pred = model(X_test)
    y_pred_numpy = y_pred.numpy()
    y_test_numpy = y_test.numpy()

print(f"R2: {r2_score(y_test_numpy, y_pred_numpy):.4f}")
print(f"MSE: {mean_squared_error(y_test_numpy, y_pred_numpy):.4f}")

R2: 0.2498
MSE: 0.0775
